In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [0]:
#initialize spark session
spark = SparkSession.builder.appName("Employee Performance Review Analysis").getOrCreate()

In [0]:
data = [
    (1, '2024-01-10', 'Engineering', 5, 'John'),
    (2, '2024-01-11', 'HR', 4, 'Jane'),
    (3, '2024-01-12', 'Sales', 3, 'Sam'),
    (4, '2024-02-01', 'Engineering', 5, 'John'),
    (1, '2024-03-10', 'Engineering', 4, 'Jane'),
    (2, '2024-03-11', 'HR', None, 'Sam')
]


In [0]:
# create dataframe

columns = ["emp_id", "review_date", "department", "rating", "reviewer"]
df = spark.createDataFrame(data,schema=columns)
df.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
df_rating_filter= df.filter(col("rating") >=4)
df_rating_filter.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
+------+-----------+-----------+------+--------+



In [0]:

#groupBy - collapse data into a shorter summary table
# Window function - calculate group metrics while keeping every original row intact.
# coalesce replaces NULL ratings with the department's average rating calculated by the window function
window_spec = Window.partitionBy("department")

df_filled = df.withColumn(
    "rating",
    coalesce(
        col("rating"),
        avg("rating").over(window_spec)
    )
)

df_filled.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|   5.0|    John|
|     4| 2024-02-01|Engineering|   5.0|    John|
|     1| 2024-03-10|Engineering|   4.0|    Jane|
|     2| 2024-01-11|         HR|   4.0|    Jane|
|     2| 2024-03-11|         HR|   4.0|     Sam|
|     3| 2024-01-12|      Sales|   3.0|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
df_no_duplicates = df.dropDuplicates(
    ["emp_id", "review_date"]
)

df_no_duplicates.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
df_selected = df.select("emp_id","review_date")
df_selected.show()

+------+-----------+
|emp_id|review_date|
+------+-----------+
|     1| 2024-01-10|
|     2| 2024-01-11|
|     3| 2024-01-12|
|     4| 2024-02-01|
|     1| 2024-03-10|
|     2| 2024-03-11|
+------+-----------+



In [0]:
#df_grouped = df.groupBy("department").agg({'rating':'avg'})
df_grouped = df.groupBy("department").agg(avg('rating').alias("dept_avg"))
df_grouped.show()

+-----------+-----------------+
| department|         dept_avg|
+-----------+-----------------+
|Engineering|4.666666666666667|
|         HR|              4.0|
|      Sales|              3.0|
+-----------+-----------------+



In [0]:
employees = [
    (1, "Jack"),
    (2, "Mary"),
    (3, "John"),
    (4, "Jane"),
    (5, "Sam"),
    (6, "Sara")
]

emp_columns = ["emp_id", "emp_name"]
df_employees = spark.createDataFrame(employees,schema=emp_columns)

In [0]:
df_joined = df.join(df_employees, on="emp_id", how="inner")
df_joined.show()

+------+-----------+-----------+------+--------+--------+
|emp_id|review_date| department|rating|reviewer|emp_name|
+------+-----------+-----------+------+--------+--------+
|     1| 2024-01-10|Engineering|     5|    John|    Jack|
|     2| 2024-01-11|         HR|     4|    Jane|    Mary|
|     3| 2024-01-12|      Sales|     3|     Sam|    John|
|     4| 2024-02-01|Engineering|     5|    John|    Jane|
|     1| 2024-03-10|Engineering|     4|    Jane|    Jack|
|     2| 2024-03-11|         HR|  NULL|     Sam|    Mary|
+------+-----------+-----------+------+--------+--------+



In [0]:
new_reviews = [
    (5, '2024-04-01', 'Sales', 5, 'John'),
    (6, '2024-04-02', 'HR', 4, 'Jane')
]

df_new_reviews = spark.createDataFrame(
    new_reviews,
    columns
)

In [0]:
#union - add rows
#join - add columns
df_union = df.union(df_new_reviews)
df_union.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
|     5| 2024-04-01|      Sales|     5|    John|
|     6| 2024-04-02|         HR|     4|    Jane|
+------+-----------+-----------+------+--------+



In [0]:
df_union.createOrReplaceTempView("performance_review")
sql_result = spark.sql("""
                     SELECT emp_id, avg(rating) AS avg_rating FROM performance_review
                       GROUP BY emp_id
                       """)

sql_result.show()

+------+----------+
|emp_id|avg_rating|
+------+----------+
|     1|       4.5|
|     2|       4.0|
|     3|       3.0|
|     4|       5.0|
|     5|       5.0|
|     6|       4.0|
+------+----------+



In [0]:
df = df.withColumn("review_date", to_date("review_date"))
df.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
window_spec= Window.partitionBy("emp_id").orderBy("review_date")
df_with_cumulative_avg = df.withColumn("cumulative_avg", avg("rating").over(window_spec))
df_with_cumulative_avg.show()

+------+-----------+-----------+------+--------+--------------+
|emp_id|review_date| department|rating|reviewer|cumulative_avg|
+------+-----------+-----------+------+--------+--------------+
|     1| 2024-01-10|Engineering|     5|    John|           5.0|
|     1| 2024-03-10|Engineering|     4|    Jane|           4.5|
|     2| 2024-01-11|         HR|     4|    Jane|           4.0|
|     2| 2024-03-11|         HR|  NULL|     Sam|           4.0|
|     3| 2024-01-12|      Sales|     3|     Sam|           3.0|
|     4| 2024-02-01|Engineering|     5|    John|           5.0|
+------+-----------+-----------+------+--------+--------------+

